In [1]:
!pip install -q transformers accelerate evaluate sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.8 MB/s eta 0:00:00


In [2]:
!pip uninstall -y datasets
!pip install datasets==3.6.0

Found existing installation: datasets 4.0.0
Uninstalling datasets-4.0.0:
  Successfully uninstalled datasets-4.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 34.6 MB/s eta 0:00:00


In [19]:
import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)

from datasets import Dataset

In [20]:
print("PyTorch Version :", torch.__version__)
print("GPU Available :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))

PyTorch Version : 2.5.1+cu121
GPU Available : True
GPU : Tesla T4


In [21]:
from google.colab import files

uploaded = files.upload()

KeyboardInterrupt: 

In [22]:
import pandas as pd

df = pd.read_csv("final_emotion_dataset.csv")

print(df.shape)
df.head()

(200235, 6)


,text,sentiment,clean_text,label,main_emotion,main_label
0,I experienced this emotion when my grandfather...,sadness,experienced emotion grandfather passed away,26,Sad,8
1,"when I first moved in , I walked everywhere ....",neutral,first moved walked everywhere within week purs...,20,Neutral,6
2,"` Oh ! "" she bleated , her voice high and rath...",anger,oh bleated voice high rather indignant,2,Angry,1
3,"However , does the right hon. Gentleman recogn...",fear,however right hon gentleman recognise profound...,14,Fear,4
4,My boyfriend didn't turn up after promising th...,sadness,boyfriend not turn promising coming,26,Sad,8


In [23]:
df = df[["clean_text", "sentiment"]]
df = df.dropna()
df = df.drop_duplicates()

df.reset_index(drop=True, inplace=True)

print(df.shape)

(197168, 2)


In [24]:
label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["sentiment"])

print("Total Classes :", len(label_encoder.classes_))

print(label_encoder.classes_)

Total Classes : 28
['admiration' 'amusement' 'anger' 'annoyance' 'approval' 'caring'
 'confusion' 'curiosity' 'desire' 'disappointment' 'disapproval' 'disgust'
 'embarrassment' 'excitement' 'fear' 'gratitude' 'grief' 'joy' 'love'
 'nervousness' 'neutral' 'optimism' 'pride' 'realization' 'relief'
 'remorse' 'sadness' 'surprise']


In [25]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

print("Training :", train_df.shape)
print("Validation :", val_df.shape)

Training : (157734, 3)
Validation : (39434, 3)


In [26]:
MODEL_NAME = "distilbert-base-uncased"
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [27]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

In [28]:
def tokenize(batch):
    return tokenizer(
        batch["clean_text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)

Map:   0%|          | 0/157734 [00:00<?, ? examples/s]

Map:   0%|          | 0/39434 [00:00<?, ? examples/s]

In [29]:
train_dataset = train_dataset.remove_columns(
    ["clean_text", "sentiment", "__index_level_0__"]
)

val_dataset = val_dataset.remove_columns(
    ["clean_text", "sentiment", "__index_level_0__"]
)

train_dataset.set_format("torch")
val_dataset.set_format("torch")

In [30]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_encoder.classes_)
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [36]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./distilbert_subemotion",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    load_best_model_at_end=True,
    logging_steps=100,
    report_to="none"
)

In [37]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, average="weighted")
    }

In [38]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

In [19]:
print('Performing a clean reinstallation of PyTorch components...')

# Uninstall all existing PyTorch, torchvision, and torchaudio packages
!pip uninstall -y torch torchvision torchaudio

# Install a stable torch and torchvision version for CUDA 12.1 and Python 3.12
# Using 2.5.1 for torch and 0.20.1 for torchvision, which were previously shown to be compatible with cu121
!pip install torch==2.5.1 torchvision==0.20.1 --index-url https://download.pytorch.org/whl/cu121

print('PyTorch components reinstalled successfully.')

Performing a clean reinstallation of PyTorch components...
Found existing installation: torch 2.11.0+cu128
Uninstalling torch-2.11.0+cu128:
  Successfully uninstalled torch-2.11.0+cu128
Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 1.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 18.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 23.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 35.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 83.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 943.3 k

PyTorch components reinstalled successfully.


In [1]:
print('Uninstalling torchaudio to resolve the import error...')
!pip uninstall -y torchaudio

Uninstalling torchaudio to resolve the import error...


In [34]:
import datasets
import torchvision
import torch

print(datasets.__version__)
print(torchvision.__version__)
print(torch.__version__)

3.6.0
0.20.1+cu121
2.5.1+cu121


In [3]:
print('Uninstalling current torchvision...')
!pip uninstall -y torchvision

Uninstalling current torchvision...
Found existing installation: torchvision 0.20.1+cu121
Uninstalling torchvision-0.20.1+cu121:
  Successfully uninstalled torchvision-0.20.1+cu121


In [4]:
print('Installing torchvision==0.20.1 compatible with torch 2.1.x and CUDA 12.1...')
!pip install torchvision==0.20.1 --index-url https://download.pytorch.org/whl/cu121

Installing torchvision==0.20.1 compatible with torch 2.1.x and CUDA 12.1...
Looking in indexes: https://download.pytorch.org/whl/cu121
  Using cached https://download-r2.pytorch.org/whl/cu121/torchvision-0.20.1%2Bcu121-cp312-cp312-linux_x86_64.whl (7.3 MB)


In [39]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.961827,1.944683,0.413273,0.353012
2,1.838997,1.924780,0.414693,0.372316


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=19718, training_loss=1.9331757140843981, metrics={'train_runtime': 4175.4294, 'train_samples_per_second': 75.553, 'train_steps_per_second': 4.722, 'total_flos': 1.0452150464606208e+16, 'train_loss': 1.9331757140843981, 'epoch': 2.0})

In [47]:
SAVE_PATH = "/content/distilbert_sub_emotion"

trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print("✅ Model saved to:", SAVE_PATH)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved to: /content/distilbert_sub_emotion


In [48]:
import joblib

joblib.dump(
    sub_encoder,
    "/content/sub_emotion_encoder.pkl"
)

print("✅ Encoder saved.")

NameError: name 'sub_encoder' is not defined

In [49]:
from google.colab import files
import shutil

# Zip the model folder
shutil.make_archive(
    "/content/distilbert_sub_emotion",
    "zip",
    "/content/distilbert_sub_emotion"
)

# Download files
files.download("/content/distilbert_sub_emotion.zip")
files.download("/content/sub_emotion_encoder.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

FileNotFoundError: Cannot find file: /content/sub_emotion_encoder.pkl